In [0]:
from pyspark.sql.functions import *

In [0]:
orders = spark.table("retail_silver.silver1_orders_clean")

customers = (
    spark.table("retail_silver.dim_customer_scd2")
    .filter(col("is_current") == True)
)

products = (
    spark.table("retail_silver.dim_product_scd2")
    .filter(col("is_current") == True)
)

stores = spark.table("retail_raw.bronze_stores")

In [0]:
print("Orders :", orders.count())
print("Customers :", customers.count())
print("Products :", products.count())
print("Stores :", stores.count())

Orders : 11498
Customers : 2816
Products : 822
Stores : 80


In [0]:
stores.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



In [0]:
from pyspark.sql.window import Window
store_window = Window.partitionBy("store_id").orderBy(col("ingestion_ts").desc())

dim_store = (
    stores
    .filter(col("store_id").isNotNull())
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("city", coalesce(trim(col("city")), lit("Unknown")))
    .withColumn("region", coalesce(trim(col("region")), lit("Unknown")))
    .withColumn("status", coalesce(trim(col("status")), lit("Unknown")))
    .withColumn("rn", row_number().over(store_window))
    .filter(col("rn") == 1)
    .drop("rn")
    .withColumn("store_sk", monotonically_increasing_id())
)

In [0]:
dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.dim_store")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_stores
FROM retail_silver.dim_store
""").show()

+------------+
|total_stores|
+------------+
|          75|
+------------+



In [0]:
customer_dim = spark.table("retail_silver.dim_customer_scd2")
product_dim = spark.table("retail_silver.dim_product_scd2")
store_dim = spark.table("retail_silver.dim_store")

In [0]:
customer_match_window = (
    Window
    .partitionBy("o.order_id")
    .orderBy(
        col("c.effective_start_date").desc(),
        col("c.ingestion_ts").desc(),
        col("c.customer_sk").desc()
    )
)

order_customer_keys = (
    orders.alias("o")
    .join(
        customer_dim.alias("c"),
        (col("o.customer_id") == col("c.customer_id")) &
        (
            to_date(col("o.order_ts")).between(
                col("c.effective_start_date"),
                col("c.effective_end_date")
            )
        ),
        "left"
    )
    .withColumn("customer_rank", row_number().over(customer_match_window))
    .filter(col("customer_rank") == 1)
    .select(
        col("o.order_id"),
        col("c.customer_sk")
    )
)

print("Resolved Customer Keys:", order_customer_keys.count())

Resolved Customer Keys: 11498


In [0]:
product_match_window = (
    Window
    .partitionBy("o.order_id")
    .orderBy(
        col("p.effective_start_date").desc(),
        col("p.ingestion_ts").desc(),
        col("p.product_sk").desc()
    )
)

order_product_keys = (
    orders.alias("o")
    .join(
        product_dim.alias("p"),
        (col("o.product_id") == col("p.product_id")) &
        (
            to_date(col("o.order_ts")).between(
                col("p.effective_start_date"),
                col("p.effective_end_date")
            )
        ),
        "left"
    )
    .withColumn("product_rank", row_number().over(product_match_window))
    .filter(col("product_rank") == 1)
    .select(
        col("o.order_id"),
        col("p.product_sk")
    )
)

print("Resolved Product Keys:", order_product_keys.count())

Resolved Product Keys: 11498


In [0]:
fact_orders = (
    orders.alias("o")
    .join(order_customer_keys.alias("ck"), "order_id", "left")
    .join(order_product_keys.alias("pk"), "order_id", "left")
    .join(
        store_dim.alias("s"),
        col("o.store_id") == col("s.store_id"),
        "left"
    )
    .select(
        col("o.order_id"),
        col("o.order_ts"),
        col("ck.customer_sk"),
        col("pk.product_sk"),
        col("s.store_sk"),
        col("o.customer_id"),
        col("o.product_id"),
        col("o.store_id"),
        col("o.quantity"),
        col("o.unit_price"),
        col("o.discount_pct"),
        col("o.gross_amount"),
        col("o.order_status")
    )
)

In [0]:
print("Orders Rows:", orders.count())
print("Fact Rows:", fact_orders.count())

Orders Rows: 11498
Fact Rows: 11498


In [0]:
fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.fact_orders")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_fact_orders
FROM retail_gold.fact_orders
""").show()

+-----------------+
|total_fact_orders|
+-----------------+
|            11498|
+-----------------+



In [0]:
gold_daily_sales = (
    fact_orders
    .withColumn("order_date", to_date(col("order_ts")))
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("gross_amount").alias("total_revenue"),
        sum("quantity").alias("total_quantity")
    )
    .orderBy("order_date")
)

In [0]:
gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_daily_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_daily_sales
ORDER BY order_date
LIMIT 10
""").show()

+----------+------------+-----------------+--------------+
|order_date|total_orders|    total_revenue|total_quantity|
+----------+------------+-----------------+--------------+
|2025-10-01|          60|7931198.779999998|           193|
|2025-10-02|          49|5833780.930000002|           135|
|2025-10-03|          57|       6742787.45|           160|
|2025-10-04|          63|        8764260.0|           201|
|2025-10-05|          52|        6296934.8|           159|
|2025-10-06|          61|8034282.170000002|           188|
|2025-10-07|          63|8086661.389999999|           174|
|2025-10-08|          63|6900786.519999997|           185|
|2025-10-09|          49|6025594.720000001|           136|
|2025-10-10|          65|       7819608.32|           208|
+----------+------------+-----------------+--------------+



In [0]:
spark.sql("""
SELECT COUNT(*) AS total_days
FROM retail_gold.gold_daily_sales
""").show()

+----------+
|total_days|
+----------+
|       191|
+----------+



In [0]:
product_lookup = (
    spark.table("retail_silver.dim_product_scd2")
    .select("product_sk", "category")
)

gold_category_sales = (
    fact_orders.alias("f")
    .join(
        product_lookup.alias("p"),
        col("f.product_sk") == col("p.product_sk"),
        "left"
    )
    .groupBy(col("p.category").alias("category"))
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        sum("f.gross_amount").alias("total_revenue"),
        sum("f.quantity").alias("total_units")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
gold_category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_category_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_category_sales
ORDER BY total_revenue DESC
""").show()

+-----------+------------+--------------------+-----------+
|   category|total_orders|       total_revenue|total_units|
+-----------+------------+--------------------+-----------+
|       Home|        2809| 3.631771093700006E8|       8837|
|    Fashion|        2786|3.5337554544999963E8|       8660|
|Electronics|        2485| 3.080710727900003E8|       7528|
|    Grocery|        2287| 2.915023583899995E8|       7060|
|     Beauty|        2258| 2.781472295299998E8|       6937|
|       NULL|         534| 6.548430620999997E7|       1590|
|    Unknown|         272|3.2920689210000005E7|        820|
+-----------+------------+--------------------+-----------+



In [0]:
gold_category_sales = (
    fact_orders.alias("f")
    .join(
        product_lookup.alias("p"),
        col("f.product_sk") == col("p.product_sk"),
        "left"
    )
    .withColumn(
        "category",
        coalesce(col("p.category"), lit("Unknown"))
    )
    .groupBy("category")
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        round(sum("f.gross_amount"), 2).alias("total_revenue"),
        sum("f.quantity").alias("total_units")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
gold_category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_category_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_category_sales
ORDER BY total_revenue DESC
""").show()

+-----------+------------+--------------+-----------+
|   category|total_orders| total_revenue|total_units|
+-----------+------------+--------------+-----------+
|       Home|        2809|3.6317710937E8|       8837|
|    Fashion|        2786|3.5337554545E8|       8660|
|Electronics|        2485|3.0807107279E8|       7528|
|    Grocery|        2287|2.9150235839E8|       7060|
|     Beauty|        2258|2.7814722953E8|       6937|
|    Unknown|         806| 9.840499542E7|       2410|
+-----------+------------+--------------+-----------+



In [0]:
gold_category_sales = (
    fact_orders.alias("f")
    .join(
        product_lookup.alias("p"),
        col("f.product_sk") == col("p.product_sk"),
        "left"
    )
    .withColumn(
        "category",
        coalesce(col("p.category"), lit("Unknown"))
    )
    .groupBy("category")
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        round(sum("f.gross_amount"), 2).alias("total_revenue"),
        sum("f.quantity").alias("total_units")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
gold_category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_category_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_category_sales
ORDER BY total_revenue DESC
""").show()

+-----------+------------+--------------+-----------+
|   category|total_orders| total_revenue|total_units|
+-----------+------------+--------------+-----------+
|       Home|        2809|3.6317710937E8|       8837|
|    Fashion|        2786|3.5337554545E8|       8660|
|Electronics|        2485|3.0807107279E8|       7528|
|    Grocery|        2287|2.9150235839E8|       7060|
|     Beauty|        2258|2.7814722953E8|       6937|
|    Unknown|         806| 9.840499542E7|       2410|
+-----------+------------+--------------+-----------+



In [0]:
customer_lookup = (
    spark.table("retail_silver.dim_customer_scd2")
    .select("customer_sk", "segment")
)

In [0]:
gold_segment_sales = (
    fact_orders.alias("f")
    .join(
        customer_lookup.alias("c"),
        col("f.customer_sk") == col("c.customer_sk"),
        "left"
    )
    .withColumn(
        "segment",
        coalesce(col("c.segment"), lit("Unknown"))
    )
    .groupBy("segment")
    .agg(
        countDistinct("f.customer_id").alias("unique_customers"),
        countDistinct("f.order_id").alias("total_orders"),
        round(sum("f.gross_amount"), 2).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
gold_segment_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_segment_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_segment_sales
ORDER BY total_revenue DESC
""").show()

+--------+----------------+------------+--------------+
| segment|unique_customers|total_orders| total_revenue|
+--------+----------------+------------+--------------+
|Platinum|             736|        3356|4.2788084804E8|
|  Silver|             730|        3339|4.2320421424E8|
| Regular|             715|        3255|4.1780917462E8|
|    Gold|             704|        3025|3.8956302565E8|
| Unknown|             283|         660| 8.145458836E7|
+--------+----------------+------------+--------------+



In [0]:
store_lookup = (
    spark.table("retail_silver.dim_store")
    .select("store_sk", "region")
)

In [0]:
gold_region_sales = (
    fact_orders.alias("f")
    .join(
        store_lookup.alias("s"),
        col("f.store_sk") == col("s.store_sk"),
        "left"
    )
    .withColumn(
        "region",
        coalesce(col("s.region"), lit("Unknown"))
    )
    .groupBy("region")
    .agg(
        countDistinct("f.order_id").alias("total_orders"),
        round(sum("f.gross_amount"), 2).alias("total_revenue"),
        sum("f.quantity").alias("total_units")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
gold_region_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_gold.gold_region_sales")

In [0]:
spark.sql("""
SELECT *
FROM retail_gold.gold_region_sales
ORDER BY total_revenue DESC
""").show()

+------+------------+--------------+-----------+
|region|total_orders| total_revenue|total_units|
+------+------------+--------------+-----------+
|Online|        2994|3.5969582191E8|       8844|
| South|        2615|3.1661056784E8|       7739|
| North|        2166|2.6897290375E8|       6523|
|  West|        2100|2.5802035052E8|       6307|
|  East|        1623|1.9396488031E8|       4826|
+------+------------+--------------+-----------+



In [0]:
spark.sql("""
SELECT 'fact_orders' AS table_name, COUNT(*) AS row_count
FROM retail_gold.fact_orders

UNION ALL

SELECT 'gold_daily_sales', COUNT(*)
FROM retail_gold.gold_daily_sales

UNION ALL

SELECT 'gold_category_sales', COUNT(*)
FROM retail_gold.gold_category_sales

UNION ALL

SELECT 'gold_segment_sales', COUNT(*)
FROM retail_gold.gold_segment_sales

UNION ALL

SELECT 'gold_region_sales', COUNT(*)
FROM retail_gold.gold_region_sales
""").show()

+-------------------+---------+
|         table_name|row_count|
+-------------------+---------+
|        fact_orders|    11498|
|   gold_daily_sales|      191|
|gold_category_sales|        6|
| gold_segment_sales|        5|
|  gold_region_sales|        5|
+-------------------+---------+

